# Concurrency

Welcome! Python offers several concurrency models: threading for I/O-bound tasks, multiprocessing for CPU-bound work, and asyncio for async I/O. This guide covers threading, multiprocessing, asyncio, the GIL, concurrent.futures, and async/await patterns.

## The Global Interpreter Lock (GIL)

The GIL allows only one thread to execute Python bytecode at a time. Threads are good for I/O-bound work (waiting on network); CPU-bound work benefits from multiprocessing.

In [ ]:
import sys
print(f"GIL: Only one thread runs Python bytecode at a time")
print(f"Use threading for I/O-bound, multiprocessing for CPU-bound")

## threading

The `threading` module provides threads. Use `Thread(target=func, args=(a, b))` and `start()`.

In [ ]:
import threading
import time

def worker(name, delay):
    print(f"{name} starting")
    time.sleep(delay)
    print(f"{name} done")

t1 = threading.Thread(target=worker, args=("A", 1))
t2 = threading.Thread(target=worker, args=("B", 0.5))
t1.start()
t2.start()
t1.join()
t2.join()
print("All threads done")

## Thread Synchronization: Lock

Use `Lock` to protect shared state from race conditions.

In [ ]:
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(10000):
        with lock:
            counter += 1

threads = [threading.Thread(target=increment) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(counter)

## multiprocessing

`multiprocessing` spawns separate processes, each with its own Python interpreter and GIL.

In [ ]:
from multiprocessing import Process, current_process

def worker(name):
    print(f"Process {current_process().name}: {name}")

if __name__ == "__main__":
    p1 = Process(target=worker, args=("A",))
    p2 = Process(target=worker, args=("B",))
    p1.start()
    p2.start()
    p1.join()
    p2.join()
    print("Done")

## concurrent.futures

`ThreadPoolExecutor` and `ProcessPoolExecutor` provide a high-level interface.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

def square(n):
    return n * n

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(square, i) for i in range(5)]
    for f in as_completed(futures):
        print(f.result())

with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(square, range(5)))
    print(results)

## asyncio Basics

asyncio provides cooperative multitasking. Use `async def` and `await`.

In [ ]:
import asyncio

async def say_hello():
    print("Hello")
    await asyncio.sleep(1)
    print("World")

async def main():
    await say_hello()

asyncio.run(main())

## Running Multiple Coroutines

Use `asyncio.gather()` to run coroutines concurrently.

In [ ]:
import asyncio

async def fetch(id, delay):
    await asyncio.sleep(delay)
    return f"Result {id}"

async def main():
    results = await asyncio.gather(
        fetch(1, 0.5),
        fetch(2, 0.3),
        fetch(3, 0.4)
    )
    print(results)

asyncio.run(main())

## async/await Patterns

Use `async with` for async context managers. Use `asyncio.create_task()` for fire-and-forget.

In [ ]:
import asyncio

async def slow_task():
    await asyncio.sleep(1)
    return "done"

async def main():
    task = asyncio.create_task(slow_task())
    print("Task started")
    result = await task
    print(result)

asyncio.run(main())

## When to Use What

| Scenario | Use |
|----------|-----|
| I/O-bound, many connections | asyncio |
| I/O-bound, simple | threading |
| CPU-bound | multiprocessing |
| Mixed | concurrent.futures or ProcessPoolExecutor |

In [ ]:
print("I/O-bound: asyncio or threading")
print("CPU-bound: multiprocessing")
print("concurrent.futures: unified API for both")

## Summary

You've learned the GIL, threading, locks, multiprocessing, concurrent.futures, asyncio, and async/await. Choose the right tool: asyncio for I/O-bound async, threading for simple I/O, multiprocessing for CPU-bound work.